# 盤前速報主播對嘴（MuseTalk 1.5）

**執行前**：選單「執行階段 → 變更執行階段類型」選 **T4 GPU**（有 L4/A100 更快）。

Google Drive 準備好這個資料夾：
```
MyDrive/finreels/
  looks/               ← 一次性：多支待機影片（不同服裝／場景／姿勢），檔名如 look01_navy-cafe.mp4
    look01_navy-cafe.mp4
    look02_white-office.mp4
    ...
  host_idle.mp4        ← 備用：looks/ 是空的或不存在時才用
  2026-09-30/voice.wav ← 每天：gemini_tts.py 產生的整段語音
```
每天的造型：第 1 格的 `LOOK` 留空 → 依日期自動輪替 looks/ 裡的影片；想指定就填檔名。
輸出：`MyDrive/finreels/2026-09-30/host.mp4` → 下載放到專案 `public/2026-09-30/host.mp4`。
待機影片的生成提示詞見專案 `docs/host_looks_prompts.md`。

第 2 格只有第一次（或 Colab 重置後）需要跑，約 10 分鐘。

In [ ]:
#@title 1. 掛載 Drive、設定日期與造型
from google.colab import drive
drive.mount('/content/drive')
import os, datetime
DATE = '2026-09-30'  #@param {type:"string"}
LOOK = ''  #@param {type:"string"}  留空＝依日期自動輪替；或填 looks/ 下的檔名（可省略 .mp4）
BASE = '/content/drive/MyDrive/finreels'
assert os.path.exists(f'{BASE}/{DATE}/voice.wav'), '找不到當天 voice.wav'

looks = sorted(f for f in os.listdir(f'{BASE}/looks') if f.lower().endswith('.mp4')) if os.path.isdir(f'{BASE}/looks') else []
if LOOK.strip():
    name = LOOK.strip() if LOOK.strip().lower().endswith('.mp4') else LOOK.strip() + '.mp4'
    assert name in looks, f'looks/ 裡找不到 {name}；現有：{looks}'
    IDLE = f'{BASE}/looks/{name}'
elif looks:
    # 日期序號 % 支數：每天換一支，全部輪完才重複；同一天重跑結果一致
    IDLE = f'{BASE}/looks/' + looks[datetime.date.fromisoformat(DATE).toordinal() % len(looks)]
else:
    IDLE = f'{BASE}/host_idle.mp4'
    assert os.path.exists(IDLE), '找不到 looks/ 也找不到 host_idle.mp4'
print('今天的待機影片：', IDLE, f'（looks/ 共 {len(looks)} 支）')

In [ ]:
#@title 2. 安裝 MuseTalk（用 uv 建 Python 3.10 環境，避開 Colab 新版 Python 與 mmcv 相容問題）
%%bash
set -e
cd /content
[ -d MuseTalk ] || git clone -q https://github.com/TMElyralab/MuseTalk.git
pip -q install uv
# 壞掉或版本不對的舊環境直接砍掉重建
[ -x /content/mt/bin/python ] && /content/mt/bin/python -c "import sys; assert sys.version_info[:2]==(3,10)" 2>/dev/null || { rm -rf /content/mt; uv venv -q --seed -p 3.10 /content/mt; }
source /content/mt/bin/activate
# Colab 可能設了 UV_SYSTEM_PYTHON=1，會讓 uv 忽略 venv 而裝到系統 Python 3.13，所以每次都明確指定 --python
unset UV_SYSTEM_PYTHON
export VIRTUAL_ENV=/content/mt
PY=/content/mt/bin/python
# 版本依 MuseTalk 官方 README / requirements.txt（torch 2.0.1+cu118、mmcv 2.0.1、mmdet 3.1.0、mmpose 1.1.0、
# huggingface_hub 0.30.2、numpy 1.23.5、transformers 4.39.2）。不要另外裝沒釘版本的套件，會蓋掉官方版本。
uv pip install -q --python $PY torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
cd MuseTalk
uv pip install -q --python $PY -r requirements.txt
# setuptools<70：新版移除了 pkg_resources，mmengine/mmpose 匯入時會找不到（官方 conda 環境附的是舊版）
uv pip install -q --python $PY openmim "setuptools<70"
uv pip install -q --python $PY mmengine
uv pip install -q --python $PY "mmcv==2.0.1" -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html
# chumpy（mmpose 的相依）的 setup.py 需要 pip，但 uv 預設的隔離建置環境沒有 → 先裝好再關閉隔離
uv pip install -q --python $PY pip "setuptools<70" wheel
uv pip install -q --python $PY --no-build-isolation chumpy==0.70
uv pip install -q --python $PY "mmdet==3.1.0" "mmpose==1.1.0"
uv pip install -q --python $PY gdown
# 釘回官方版本（huggingface-cli 已含在 huggingface_hub 內）
uv pip install -q --python $PY "huggingface_hub==0.30.2" "numpy==1.23.5" "setuptools<70"

# 權重：官方 download_weights.sh 有三個問題，先用 sed 修掉再執行：
#  1. 會自己 `pip install -U huggingface_hub[cli]`（升到 2.0 而弄壞 huggingface-cli）→ 刪掉 pip install 行
#  2. 改用中國鏡像 hf-mirror.com → 刪掉 HF_ENDPOINT
#  3. `gdown --id ...`：新版 gdown 已移除 --id，直接給 ID 即可
# 只要缺任何一個檔案就重新執行（huggingface-cli 會跳過已完整的檔案）。
NEED="musetalk/pytorch_model.bin musetalkV15/unet.pth musetalkV15/musetalk.json sd-vae/diffusion_pytorch_model.bin whisper/pytorch_model.bin dwpose/dw-ll_ucoco_384.pth syncnet/latentsync_syncnet.pt face-parse-bisent/79999_iter.pth face-parse-bisent/resnet18-5c106cde.pth"
MISSING=0; for f in $NEED; do [ -s "models/$f" ] || { echo "缺少 models/$f"; MISSING=1; }; done
if [ $MISSING = 1 ]; then
  sed -e '/^pip install/d' -e '/HF_ENDPOINT/d' -e 's/gdown --id /gdown /' download_weights.sh > /content/dw.sh
  bash /content/dw.sh
fi
for f in $NEED; do [ -s "models/$f" ] || { echo "下載後仍缺少 models/$f"; exit 1; }; done

# 檢查相依衝突（僅顯示，不中斷）
$PY -m pip check || true
ls models

In [ ]:
#@title 3. 對嘴（每天跑這格）
%%bash -s "$DATE" "$BASE" "$IDLE"
set -e
DATE=$1; BASE=$2; IDLE=$3
source /content/mt/bin/activate
# Colab kernel 的 MPLBACKEND 指向 matplotlib_inline，venv 裡沒有這個模組 → 改用無介面的 Agg
export MPLBACKEND=Agg
cd /content/MuseTalk
mkdir -p /content/work
# 待機影片轉 25fps（MuseTalk 預設），語音轉 16k mono
# -nostdin：避免 ffmpeg 吃掉 %%bash 從 stdin 傳入的後續腳本
ffmpeg -nostdin -v error -y -i "$IDLE" -r 25 -an -c:v libx264 -pix_fmt yuv420p /content/work/idle.mp4
ffmpeg -nostdin -v error -y -i "$BASE/$DATE/voice.wav" -ar 16000 -ac 1 /content/work/voice.wav
cat > /content/work/task.yaml <<YAML
task_0:
  video_path: "/content/work/idle.mp4"
  audio_path: "/content/work/voice.wav"
YAML
rm -rf /content/work/out
python -m scripts.inference \
  --inference_config /content/work/task.yaml \
  --result_dir /content/work/out \
  --unet_model_path models/musetalkV15/unet.pth \
  --unet_config models/musetalkV15/musetalk.json \
  --version v15 --ffmpeg_path /usr/bin < /dev/null
OUT=$(find /content/work/out -name "*.mp4" | head -1)
echo "MuseTalk 輸出：$OUT"
cp "$OUT" "$BASE/$DATE/host.mp4"
ffprobe -v error -show_entries format=duration -of csv=p=0 "$BASE/$DATE/host.mp4"

In [ ]:
#@title 4. 預覽
from IPython.display import Video
Video(f'{BASE}/{DATE}/host.mp4', embed=True, width=300)

### 常見問題
- **嘴型偏移／下巴模糊**：待機影片臉要正、光線均勻、不要有手或麥克風擋到嘴。
- **mmcv 安裝失敗**：多半是 Colab 換了 CUDA 版本，把第 2 格的 `cu118/torch2.0` 依 [OpenMMLab 安裝頁](https://mmcv.readthedocs.io/en/latest/get_started/installation.html) 換成對應版本。
- **時長對不上**：host.mp4 長度應等於 voice.wav；MuseTalk 會自動循環待機影片，所以待機影片短也沒關係。